# Layer 3b: Velocity Valley Detection

Validates `velocity_detector.py` in isolation — no keyposes.json required.

**Pipeline**: extract joint angles → compute angular velocity → Gaussian smooth → argrelmin

**Pass condition**: all 19 movement completions (bootstrap boundary frames)
have at least one valley within ±15 frames.

In [ ]:
import sys
import pathlib

project_root = pathlib.Path().resolve().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"project_root: {project_root}")

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt

from itf_analysis.segmentation.segmentor import load_poses_from_json, segment_video
from itf_analysis.normalization.normalizer import normalize_pose, extract_joint_angles
from itf_analysis.segmentation.velocity_detector import (
    compute_motion_velocity,
    smooth_velocity,
    find_velocity_valleys,
)

POSES_PATH = str(
    project_root / "itf_analysis" / "sample_videos" / "chon_ji_master_poses.json"
)
MATCH_TOLERANCE = 15   # frames — valley within ±15 of keypose counts as matched
SIGMA_SECONDS   = 0.1
MIN_GAP_SEC     = 0.5

## Step 1: Load poses, extract angles, estimate fps

In [ ]:
frames = load_poses_from_json(POSES_PATH)
print(f"Total frames: {len(frames)}")

fps = 1000.0 / (frames[1].timestamp_ms - frames[0].timestamp_ms) if len(frames) > 1 else 30.0
print(f"Estimated fps: {fps:.1f}")

all_frame_angles = []
skipped = []
for frame in frames:
    norm = normalize_pose(frame.landmarks)
    if norm is None:
        skipped.append(frame.frame_index)
        continue
    all_frame_angles.append((frame.frame_index, extract_joint_angles(norm)))

frame_indices = [fi for fi, _ in all_frame_angles]
print(f"Angles extracted: {len(all_frame_angles)} frames  (skipped: {skipped})")

## Step 2: Bootstrap keypose positions (from velocity segmentor)

These are the 19 boundary frames detected by the existing velocity segmentor.
Used as ground-truth positions to evaluate how well `velocity_detector.py` aligns.

In [ ]:
with warnings.catch_warnings(record=True):
    warnings.simplefilter("always")
    boundaries = segment_video(
        student_frames=frames,
        keyposes_path="nonexistent.json",
        master_total_frames=len(frames),
    )

keypose_frames = [b.boundary_frame for b in boundaries]
print(f"Bootstrap keypose frames ({len(keypose_frames)}):")
print(keypose_frames)

## Step 3: Compute motion velocity

Sum of absolute angle changes across all joints between consecutive frames.

In [ ]:
velocity = compute_motion_velocity(all_frame_angles, fps)
print(f"Velocity array length: {len(velocity)}  (expected N-1 = {len(all_frame_angles)-1})")
print(f"Min: {velocity.min():.2f}  Max: {velocity.max():.2f}  Mean: {velocity.mean():.2f} deg/frame")

# x-axis: associate velocity[i] with frame_indices[i]
vel_frames = np.array(frame_indices[:len(velocity)])

## Plot 1: Raw motion velocity time series

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(vel_frames, velocity, linewidth=0.6, color="steelblue", alpha=0.8, label="raw velocity")

for kf in keypose_frames:
    ax.axvline(kf, color="green", linewidth=0.8, alpha=0.5)
ax.axvline(keypose_frames[0], color="green", linewidth=0.8, alpha=0.5,
           label="bootstrap keypose")

ax.set_xlabel("frame index")
ax.set_ylabel("total angle change (deg/frame)")
ax.set_title("Raw angular motion velocity — Chon-Ji master")
ax.legend()
plt.tight_layout()
plt.show()

## Plot 2: Smoothing comparison (sigma sweep)

Compare raw vs. three sigma values to find the best noise suppression
without flattening genuine velocity valleys.

In [ ]:
sigmas = [0.05, 0.1, 0.2]
smoothed_variants = {s: smooth_velocity(velocity, fps, sigma_seconds=s) for s in sigmas}

fig, axes = plt.subplots(len(sigmas) + 1, 1, figsize=(16, 12), sharex=True)

axes[0].plot(vel_frames, velocity, linewidth=0.6, color="gray", alpha=0.8)
for kf in keypose_frames:
    axes[0].axvline(kf, color="green", linewidth=0.8, alpha=0.4)
axes[0].set_title("Raw velocity")
axes[0].set_ylabel("deg/frame")

for i, s in enumerate(sigmas):
    sv = smoothed_variants[s]
    order = max(1, int(MIN_GAP_SEC * fps))
    (vi,) = __import__('scipy.signal', fromlist=['argrelmin']).argrelmin(sv, order=order)
    axes[i+1].plot(vel_frames, sv, linewidth=0.8, color="steelblue", alpha=0.85,
                   label=f"sigma={s}s  ({len(vi)} valleys)")
    axes[i+1].plot(vel_frames[vi], sv[vi], "rv", markersize=5, label="valleys")
    for kf in keypose_frames:
        axes[i+1].axvline(kf, color="green", linewidth=0.8, alpha=0.4)
    axes[i+1].set_title(f"Smoothed  sigma={s}s")
    axes[i+1].set_ylabel("deg/frame")
    axes[i+1].legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("frame index")
plt.suptitle("Smoothing sigma comparison (green = bootstrap keypose positions)", y=1.01)
plt.tight_layout()
plt.show()

## Step 4: Detect valleys with chosen sigma

In [ ]:
smoothed = smooth_velocity(velocity, fps, sigma_seconds=SIGMA_SECONDS)
valleys  = find_velocity_valleys(smoothed, fps, min_gap_sec=MIN_GAP_SEC)

valley_frames = np.array([frame_indices[v] for v in valleys])
print(f"Sigma: {SIGMA_SECONDS}s ({SIGMA_SECONDS*fps:.1f} frames)")
print(f"Min gap: {MIN_GAP_SEC}s ({MIN_GAP_SEC*fps:.0f} frames)")
print(f"Valleys detected: {len(valleys)}  (need >= 19)")
print(f"Valley frame positions: {valley_frames.tolist()}")

## Plot 3: Detected valleys vs keypose positions

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

# Top: smoothed velocity with valleys and keyposes
axes[0].plot(vel_frames, smoothed, linewidth=0.9, color="steelblue", alpha=0.85,
             label="smoothed velocity")
axes[0].plot(valley_frames, smoothed[valleys], "rv", markersize=6,
             label=f"{len(valleys)} valleys")
for kf in keypose_frames:
    axes[0].axvline(kf, color="green", linewidth=1.0, alpha=0.6)
axes[0].axvline(keypose_frames[0], color="green", linewidth=1.0, alpha=0.6,
                label="bootstrap keypose")
axes[0].set_ylabel("angular velocity (deg/frame)")
axes[0].set_title("Smoothed velocity with detected valleys and bootstrap keyposes")
axes[0].legend()

# Bottom: movement label bands
colors_alt = ["#e8f4fd", "#fef9e7"]
for i, b in enumerate(boundaries):
    axes[1].axvspan(b.start_frame, b.end_frame, alpha=0.4,
                    color=colors_alt[i % 2])
    axes[1].text((b.start_frame + b.end_frame) / 2, 0.5, str(b.movement_index),
                 ha="center", va="center", fontsize=8, color="#333")
axes[1].plot(valley_frames, [1] * len(valley_frames), "rv", markersize=6,
             label="valleys")
axes[1].plot(keypose_frames, [0] * len(keypose_frames), "g^", markersize=6,
             label="keyposes")
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(["keypose", "valley"])
axes[1].set_xlabel("frame index")
axes[1].set_title("Valley (▼) vs keypose (▲) alignment per movement")
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.show()

## Step 5: Quantitative alignment evaluation

For each of 19 keyposes, find the nearest valley and measure the offset.
Negative delta = valley fires BEFORE completion (normal: body slows before settling).
Positive delta = valley fires AFTER.

In [ ]:
print(f"{'Mov':>4} {'keypose':>8} {'nearest_v':>10} {'delta':>7} {'|delta|':>8} {'match':>7}")
print("-" * 52)

match_count = 0
deltas = []

for b in boundaries:
    kf = b.boundary_frame
    if len(valley_frames) == 0:
        nearest_v, delta = None, None
        matched = False
    else:
        diffs = valley_frames - kf
        closest_idx = int(np.argmin(np.abs(diffs)))
        nearest_v = int(valley_frames[closest_idx])
        delta = int(diffs[closest_idx])
        matched = abs(delta) <= MATCH_TOLERANCE

    if matched:
        match_count += 1
    if delta is not None:
        deltas.append(delta)

    print(
        f"{b.movement_index:>4} {kf:>8} {str(nearest_v):>10} "
        f"{str(delta):>7} {str(abs(delta) if delta is not None else '—'):>8} "
        f"{'OK' if matched else 'MISS':>7}"
    )

print(f"\nMatched (|delta| <= {MATCH_TOLERANCE} frames): {match_count} / 19")
if deltas:
    print(f"Delta range : {min(deltas):+d} ~ {max(deltas):+d} frames")
    print(f"Mean delta  : {np.mean(deltas):+.1f}  (negative = valley before keypose)")
    print(f"Median |delta|: {np.median(np.abs(deltas)):.1f} frames")

In [ ]:
# Tolerance sweep: how many movements matched at different tolerances?
tolerances = [5, 10, 15, 20, 30]
print(f"{'Tolerance':>12} {'Matched':>10} {'%':>8}")
print("-" * 34)
for tol in tolerances:
    n = sum(1 for d in deltas if abs(d) <= tol)
    print(f"{tol:>10} fr {n:>8} / 19  {n/19*100:>6.1f}%")

## Plot 4: Delta distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

mov_nums = [b.movement_index for b in boundaries]
axes[0].bar(mov_nums, deltas,
            color=["steelblue" if abs(d) <= MATCH_TOLERANCE else "red" for d in deltas],
            edgecolor="white")
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].axhline(MATCH_TOLERANCE, color="orange", linestyle="--",
                linewidth=1, label=f"+{MATCH_TOLERANCE} fr")
axes[0].axhline(-MATCH_TOLERANCE, color="orange", linestyle="--",
                linewidth=1, label=f"-{MATCH_TOLERANCE} fr")
axes[0].set_xlabel("Movement index")
axes[0].set_ylabel("delta (valley - keypose) frames")
axes[0].set_title("Valley offset per movement  (blue=matched, red=missed)")
axes[0].set_xticks(mov_nums)
axes[0].legend()

axes[1].hist(deltas, bins=20, color="steelblue", edgecolor="white")
axes[1].axvline(0, color="green", linewidth=1.2, label="exact")
axes[1].axvline(np.mean(deltas), color="red", linestyle="--",
                label=f"mean={np.mean(deltas):+.1f}")
axes[1].set_xlabel("delta (frames)")
axes[1].set_ylabel("count")
axes[1].set_title("Delta distribution")
axes[1].legend()

plt.tight_layout()
plt.show()

## Plot 5: Zoomed view — valleys vs keyposes per movement

Each panel shows one movement window with its velocity curve,
keypose position, and all valleys within range.

In [ ]:
ncols, nrows = 4, 5
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 16))
axes = axes.flatten()
vel_arr = np.array(frame_indices[:len(smoothed)])

for i, b in enumerate(boundaries):
    ax = axes[i]
    lo = b.start_frame
    hi = b.end_frame

    window_mask = (vel_arr >= lo) & (vel_arr <= hi)
    ax.plot(vel_arr[window_mask], smoothed[window_mask],
            linewidth=1.0, color="steelblue")

    ax.axvline(b.boundary_frame, color="green", linewidth=1.5, label=f"kp={b.boundary_frame}")

    in_window = valley_frames[(valley_frames >= lo) & (valley_frames <= hi)]
    if len(in_window):
        in_window_pos = [valleys[np.where(valley_frames == vf)[0][0]] for vf in in_window]
        ax.plot(in_window, smoothed[in_window_pos], "rv", markersize=6)

    ax.set_title(f"Mov {b.movement_index}", fontsize=9)
    ax.set_ylabel("vel", fontsize=7)
    ax.legend(fontsize=6)

for j in range(len(boundaries), len(axes)):
    axes[j].set_visible(False)

fig.supxlabel("frame index")
plt.suptitle("Per-movement velocity window  (green=keypose, red▼=valley)", y=1.01)
plt.tight_layout()
plt.show()

## Validation Checklist

- [ ] Velocity array length = N-1
- [ ] Valleys detected >= 19
- [ ] All 19 keyposes matched within ±15 frames (tolerance sweep)
- [ ] Mean delta < 0 (valleys slightly before completion — physically expected)
- [ ] Zoomed plots show clear valley near each movement's keypose

**Note on false positives**: Valley count > 19 is normal and expected.
The combiner (03c) will use search windows to select one valley per movement
and discard extras. The key requirement here is that every keypose has
at least one valley within ±15 frames.

In [ ]:
all_matched = all(abs(d) <= MATCH_TOLERANCE for d in deltas)
print("=" * 45)
print(f"Valleys >= 19        : {'PASS' if len(valleys) >= 19 else 'FAIL'}  ({len(valleys)} found)")
print(f"All keyposes matched : {'PASS' if all_matched else 'FAIL'}  ({match_count}/19 within {MATCH_TOLERANCE} frames)")
print(f"Mean delta           : {np.mean(deltas):+.1f} frames  ({'physically expected' if np.mean(deltas) < 0 else 'check alignment'})")
print("=" * 45)

if not all_matched:
    missed = [b.movement_index for b, d in zip(boundaries, deltas) if abs(d) > MATCH_TOLERANCE]
    print(f"Missed movements: {missed}")
    print("Try increasing sigma_seconds or decreasing min_gap_sec.")